### Silver Layer- Data Transformation

1. Load bronze tables and convert string timestamps into proper timestamp formats for analysis

In [0]:
from pyspark.sql.functions import col, to_timestamp, hour, minute, date_format

df_trips = spark.table('taxi_catalog.taxi_project_schema.bronze_trips_table')


In [0]:
df_zones = spark.table('taxi_catalog.taxi_project_schema.bronze_zones_table')

2. Clean data by removing rows with null values, extract time-based features, and enrich trip data by joining with zone details.

In [0]:
# convert to proper timestamp

df_trips = df_trips.withColumn(
    "pickup_timestamp",
    to_timestamp("tpep_pickup_datetime", "yy/MM/dd H:mm")
    ).withColumn(
        "dropoff_time",
        to_timestamp("tpep_dropoff_datetime", "yy/MM/dd H:mm")
)

In [0]:
# convert to proper timestamp

df_trips = df_trips.withColumn(
    "pickup_time",
    to_timestamp("tpep_pickup_datetime", "yy/MM/dd H:mm")
    ).withColumn(
        "dropoff_time",
        to_timestamp("tpep_dropoff_datetime", "yy/MM/dd H:mm")
)

Clean the data

In [0]:
df_trips_clean = df_trips.filter(
    col("pickup_time").isNotNull() & 
    col("dropoff_time").isNotNull() &
    col("PULocationID").isNotNull() &
    col("DOLocationID").isNotNull() &
    (col("trip_distance") > 0) &
    (col("total_amount") > 0)
)

Extract only time features

In [0]:
df_trips_clean = df_trips_clean\
    .withColumn("pickup_hour", hour("pickup_time"))\
    .withColumn("pickup_minute", minute("pickup_time"))\
    .withColumn("dropoff_hour", hour("dropoff_time"))\
    .withColumn("pickup_time_only",date_format("pickup_time", "H:mm:ss"))\
    .withColumn("dropoff_time_only",date_format("dropoff_time", "H:mm:ss"))


In [0]:
from pyspark.sql.functions import col

df_enriched = df_trips_clean.alias("trips")\
    .join(
        df_zones.alias("pu"),
        col("trips.PULocationID") == col("pu.LocationID")
    )\
    .join(
        df_zones.alias("do"),
        col("trips.DOLocationID") == col("do.LocationID"),
        "left"
    )

In [0]:
df_final = df_enriched.select(
    col("trips.*"),
    col("pu.ZoneName").alias("pickup_zone"),
    col("pu.Latitude").alias("pickup_latitude"),
    col("pu.Longitude").alias("pickup_longitude"),

    col("do.ZoneName").alias("dropoff_zone"),
    col("do.Latitude").alias("dropoff_latitude"),
    col("do.Longitude").alias("dropoff_longitude")
)

df_final.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema","true")\
    .saveAsTable("taxi_catalog.taxi_project_schema.silver_trips_enriched")

In [0]:
display(df_final)